# Auditoría de etiquetas y balance de clases

Este notebook analiza localmente `data/raw/train_labels.csv` **sin modificarlo**. Está separado del modelamiento para revisar primero cantidad de elementos, estructura, faltantes, duplicados, balance de clases y correspondencia con los NIfTI disponibles.

> No muestra filas individuales ni abre imágenes. Antes de compartir o versionar el notebook ejecutado, usa **Clear All Outputs**.

In [4]:
from hashlib import sha256
from pathlib import Path
from zipfile import ZipFile

import ipywidgets as widgets
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

LABELS_PATH = PROJECT_ROOT / 'data' / 'raw' / 'train_labels.csv'
NIFTI_ARCHIVE = PROJECT_ROOT / 'data' / 'raw' / 'niftis.zip'

if not LABELS_PATH.exists():
    raise FileNotFoundError(f'No se encontró: {LABELS_PATH}')

labels = pd.read_csv(LABELS_PATH)
labels_hash = sha256(LABELS_PATH.read_bytes()).hexdigest()

display(Markdown(
    f'**Fuente:** `{LABELS_PATH.name}`  \n'
    f'**Filas:** `{len(labels):,}` · **Columnas:** `{labels.shape[1]:,}`  \n'
    f'**SHA-256:** `{labels_hash}`'
))

**Fuente:** `train_labels.csv`  
**Filas:** `1,362` · **Columnas:** `2`  
**SHA-256:** `1660731b5d51b1f91b021c41779478722139b487d44a3e25b7f54800c16ef75e`

## 1. Estructura y calidad básica

In [5]:
schema_summary = pd.DataFrame({
    'columna': labels.columns,
    'tipo': [str(labels[column].dtype) for column in labels.columns],
    'no_nulos': [int(labels[column].notna().sum()) for column in labels.columns],
    'faltantes': [int(labels[column].isna().sum()) for column in labels.columns],
    'porcentaje_faltante': [float(labels[column].isna().mean()) for column in labels.columns],
    'valores_unicos': [int(labels[column].nunique(dropna=True)) for column in labels.columns],
})

quality_summary = pd.DataFrame({
    'indicador': [
        'Filas totales',
        'Columnas totales',
        'Filas completamente duplicadas',
        'Celdas faltantes',
    ],
    'valor': [
        len(labels),
        labels.shape[1],
        int(labels.duplicated().sum()),
        int(labels.isna().sum().sum()),
    ],
})

display(Markdown('### Resumen general'))
display(quality_summary.style.format({'valor': '{:,.0f}'}).hide(axis='index'))
display(Markdown('### Diccionario observado'))
display(schema_summary.style.format({'porcentaje_faltante': '{:.1%}'}).hide(axis='index'))

### Resumen general

indicador,valor
Filas totales,"1,362"
Columnas totales,2
Filas completamente duplicadas,0
Celdas faltantes,0


### Diccionario observado

columna,tipo,no_nulos,faltantes,porcentaje_faltante,valores_unicos
uid,str,1362,0,0.0%,1362
is_pathologic,float64,1362,0,0.0%,2


## 2. Balance de clases

Selecciona la columna objetivo. El notebook propone una opción usando nombres habituales y cardinalidad baja, pero la selección queda visible para que puedas corregirla.

In [ ]:
def guess_column(columns: list[str], keywords: tuple[str, ...]) -> str | None:
    lowered = {column: column.lower() for column in columns}
    for keyword in keywords:
        exact = [column for column, value in lowered.items() if value == keyword]
        if exact:
            return exact[0]
    for keyword in keywords:
        partial = [column for column, value in lowered.items() if keyword in value]
        if partial:
            return partial[0]
    return None


def guess_target(frame: pd.DataFrame) -> str:
    preferred = guess_column(
        list(frame.columns),
        ('label', 'target', 'class', 'diagnosis', 'abnormal', 'status'),
    )
    if preferred is not None:
        return preferred

    candidates = [
        column for column in frame.columns
        if 2 <= frame[column].nunique(dropna=True) <= 20
    ]
    return candidates[-1] if candidates else frame.columns[-1]


def guess_identifier(frame: pd.DataFrame) -> str | None:
    return guess_column(
        list(frame.columns),
        ('uid', 'scan_id', 'image_id', 'filename', 'file', 'id'),
    )


def normalize_identifier(value: object) -> str:
    name = Path(str(value)).name.strip().lower()
    if name.endswith('.nii.gz'):
        return name[:-7]
    if name.endswith('.nii'):
        return name[:-4]
    return name


def imbalance_interpretation(ratio: float) -> str:
    if not np.isfinite(ratio):
        return 'No calculable'
    if ratio < 1.5:
        return 'Bajo'
    if ratio < 3.0:
        return 'Moderado'
    return 'Alto'


target_selector = widgets.Dropdown(
    options=list(labels.columns),
    value=guess_target(labels),
    description='Objetivo:',
    layout=widgets.Layout(width='480px'),
    style={'description_width': '90px'},
)
guessed_id = guess_identifier(labels)
id_options = [('Sin columna ID', None)] + [(column, column) for column in labels.columns]
id_selector = widgets.Dropdown(
    options=id_options,
    value=guessed_id,
    description='ID:',
    layout=widgets.Layout(width='480px'),
    style={'description_width': '90px'},
)
analyze_button = widgets.Button(
    description='Analizar etiquetas',
    button_style='primary',
    icon='bar-chart',
)
analysis_output = widgets.Output()


def analyze_labels(_button: widgets.Button) -> None:
    with analysis_output:
        analysis_output.clear_output(wait=True)
        target = target_selector.value
        id_column = id_selector.value

        class_values = labels[target].astype('string').fillna('<FALTANTE>')
        counts = class_values.value_counts(dropna=False).rename_axis('clase').reset_index(name='cantidad')
        counts['porcentaje'] = counts['cantidad'] / counts['cantidad'].sum()

        majority = int(counts['cantidad'].max()) if len(counts) else 0
        minority = int(counts.loc[counts['cantidad'] > 0, 'cantidad'].min()) if majority else 0
        ratio = majority / minority if minority else float('inf')
        baseline = majority / len(labels) if len(labels) else float('nan')

        metrics = pd.DataFrame({
            'métrica': [
                'Elementos etiquetados',
                'Elementos sin etiqueta',
                'Número de clases observadas',
                'Clase mayoritaria',
                'Clase minoritaria',
                'Razón mayoritaria / minoritaria',
                'Accuracy del baseline mayoritario',
                'Nivel heurístico de desbalance',
            ],
            'valor': [
                int(labels[target].notna().sum()),
                int(labels[target].isna().sum()),
                int(labels[target].nunique(dropna=True)),
                majority,
                minority,
                round(ratio, 3) if np.isfinite(ratio) else 'No calculable',
                f'{baseline:.1%}',
                imbalance_interpretation(ratio),
            ],
        })

        display(Markdown(f'### Resultado para `{target}`'))
        display(metrics.style.hide(axis='index'))
        display(Markdown('### Distribución por clase'))
        display(counts.style.format({'cantidad': '{:,.0f}', 'porcentaje': '{:.1%}'}).hide(axis='index'))

        figure = px.bar(
            counts,
            x='clase',
            y='cantidad',
            color='clase',
            text=counts['porcentaje'].map(lambda value: f'{value:.1%}'),
            title=f'Balance de clases · {target}',
            labels={'clase': 'Clase', 'cantidad': 'Cantidad'},
            template='plotly_white',
        )
        figure.update_traces(textposition='outside', cliponaxis=False)
        figure.update_layout(showlegend=False, height=480, margin=dict(t=70, r=20, b=40, l=60))
        figure.show(config={'displayModeBar': True, 'responsive': True})

        if id_column is not None:
            duplicate_ids = int(labels[id_column].duplicated(keep=False).sum())
            id_metrics = [
                ('IDs no nulos', int(labels[id_column].notna().sum())),
                ('IDs únicos', int(labels[id_column].nunique(dropna=True))),
                ('Filas involucradas en IDs duplicados', duplicate_ids),
            ]

            if NIFTI_ARCHIVE.exists():
                with ZipFile(NIFTI_ARCHIVE) as archive:
                    image_ids = {
                        normalize_identifier(name) for name in archive.namelist()
                        if name.lower().endswith(('.nii', '.nii.gz')) and not name.endswith('/')
                    }
                label_ids = {
                    normalize_identifier(value)
                    for value in labels[id_column].dropna()
                }
                id_metrics.extend([
                    ('NIfTI dentro del ZIP', len(image_ids)),
                    ('Etiquetas sin NIfTI correspondiente', len(label_ids - image_ids)),
                    ('NIfTI sin etiqueta correspondiente', len(image_ids - label_ids)),
                ])

            display(Markdown(f'### Integridad del identificador `{id_column}`'))
            display(pd.DataFrame(id_metrics, columns=['indicador', 'valor']).style.hide(axis='index'))


analyze_button.on_click(analyze_labels)
display(widgets.VBox([target_selector, id_selector, analyze_button]), analysis_output)

Output()

## 3. Cómo interpretar el resultado

- La **razón mayoritaria/minoritaria** vale 1 cuando las clases tienen el mismo tamaño.
- El nivel bajo/moderado/alto es una guía descriptiva, no una regla de modelamiento.
- La **accuracy del baseline mayoritario** indica el desempeño trivial de predecir siempre la clase más frecuente.
- Si existen IDs duplicados, etiquetas sin imagen o imágenes sin etiqueta, deben revisarse antes de definir particiones de entrenamiento.
- Este notebook no remuestrea, no elimina filas y no modifica etiquetas.